In [14]:
!pip install torch transformers datasets peft matplotlib scikit-learn

In [15]:
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import time
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "imdb"
TARGET_MODULES = ["q_lin", "v_lin", "k_lin"]
LORA_RANK = 16
MAX_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 1
BATCH_SIZE = 32

In [17]:
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
def load_and_prepare_data():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

    dataset = load_dataset(DATASET_NAME)
    tokenized_data = dataset.map(tokenize_fn, batched=True)
    train_data = tokenized_data["train"].shuffle(seed=42).select(range(2000))
    eval_data = tokenized_data["test"].shuffle(seed=42).select(range(500))
    return train_data, eval_data

In [ ]:
def get_lora_ft_args():
    return TrainingArguments(
        output_dir="./lora_ft_results",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE*2,
        learning_rate=1e-4,
        num_train_epochs=MAX_EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=5,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        weight_decay=0.01,
        report_to="none",
        fp16=True,
        gradient_accumulation_steps=2,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1
    )

In [ ]:
def get_lora_config():
    return LoraConfig(
        r=LORA_RANK,
        lora_alpha=32,
        target_modules=TARGET_MODULES,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
        inference_mode=False
    )

In [ ]:
import torch.nn.functional as F

def compute_metrics(pred):
    labels = pred.label_ids
    logits = torch.tensor(pred.predictions)
    labels_tensor = torch.tensor(labels)
    loss = F.cross_entropy(logits, labels_tensor).item()
    preds = logits.argmax(dim=-1).cpu().numpy()
    accuracy = accuracy_score(labels, preds)
    return {
        "eval_loss": loss,
        "accuracy": accuracy
    }

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_threshold=0.001
)

In [ ]:
def run_lora_finetuning(train_data, eval_data):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
    lora_config = get_lora_config()
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    args = get_lora_ft_args()
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
        callbacks=[early_stopping]
    )
    start_time = time.time()
    trainer.train()
    ft_time = time.time() - start_time
    eval_results = trainer.evaluate()
    return {
        "time": ft_time / 60,
        "accuracy": eval_results["eval_accuracy"],
        "memory": torch.cuda.max_memory_allocated() / (1024 ** 3),
        "epochs": trainer.state.epoch
    }

In [ ]:
torch.cuda.empty_cache()
train_data, eval_data = load_and_prepare_data()

In [24]:
print("Running Optimized LoRA Fine-Tuning...")
lora_results = run_lora_finetuning(train_data, eval_data)

Running Optimized LoRA Fine-Tuning...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 1,034,498 || all params: 67,989,508 || trainable%: 1.5216


Epoch,Training Loss,Validation Loss,Accuracy
1,0.642100,0.623340,0.754000


Epoch,Training Loss,Validation Loss,Accuracy
1,0.561000,0.560597,0.832000


In [25]:
print("\nResults:")
for metric in ["time", "accuracy"]:
    print(f"{metric:<15} | {lora_results[metric]:<10.2f}")


Results:
time            | 72.94     
accuracy        | 0.83      
